LangChain + OpenAI Embedding + ChromaDB

In [1]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from dotenv import load_dotenv
from pathlib import Path
import os
import shutil

/home/dmin/miniconda3/envs/apiedu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sentences = ['고양이가 창가에서 햇볕을 쬐고 있다',
 '새끼 고양이가 실타래를 가지고 논다',
 '강아지가 공원에서 뛰어다닌다',
 '강아지가 주인을 보고 꼬리를 흔든다',
 '호랑이가 숲속을 걸어 다닌다',
 '사자가 초원에서 사냥을 준비한다',
 '코끼리가 긴 코로 나뭇가지를 잡는다',
 '기린이 나무 위 잎을 먹는다',
 '원숭이가 바나나를 먹는다',
 '펭귄이 얼음 위에서 걸어간다',
 '사과가 빨갛게 익었다',
 '바나나가 노랗게 잘 익어 달콤하다',
 '포도가 달콤하고 신선하다',
 '빵이 오븐에서 갓 구워졌다',
 '라면이 뜨겁게 끓고 있다',
 '김치찌개에서 얼큰한 냄새가 난다',
 '초밥이 접시에 예쁘게 놓여 있다',
 '피자가 치즈로 가득 덮여 있다',
 '햄버거에 고기 패티가 두껍다',
 '커피 향이 방 안에 가득 퍼졌다',
 '서울의 남산타워가 빛나고 있다',
 '한강 다리 위에서 야경을 본다',
 '제주도의 바닷가가 파랗게 빛난다',
 '부산 해운대 해수욕장에 사람이 많다',
 '경복궁의 전통 건물이 웅장하다',
 '뉴욕의 자유의 여신상이 우뚝 서 있다',
 '파리의 에펠탑이 밤에 반짝인다',
 '런던의 빅벤 시계탑이 울린다',
 '도쿄의 시부야 거리가 붐빈다',
 '로마의 콜로세움이 고대의 흔적을 보여준다']

In [3]:
ai_key = os.getenv("OPENAI_API_KEY")

def create_docs(sentences_list: list[str]) -> list[Document]:
    '''
    문자열 문장을 랭체인 Document 객체로 변환
    각 문서의 메타데이터 자동 추가
    - id: 문서번호
    - source: 문서 출처
    '''

    documents = []

    for index, sentence in enumerate(sentences_list, start=1):
        # 공백만 있는 데이터는 생략
        if not sentence.strip():
            continue

        document = Document(
            page_content=sentence.strip(),
            metadata={
                "id": index,
                "source": "loc_example"
            }
        )
        documents.append(document)
    return documents

In [4]:
# 원문장을 랭채인 Document 객체로 변환
documents = create_docs(sentences)

for doc in documents:
    print(f'내용: {doc.page_content}')
    print(f'메타데이터: {doc.metadata}')
    print('='*20)

내용: 고양이가 창가에서 햇볕을 쬐고 있다
메타데이터: {'id': 1, 'source': 'loc_example'}
내용: 새끼 고양이가 실타래를 가지고 논다
메타데이터: {'id': 2, 'source': 'loc_example'}
내용: 강아지가 공원에서 뛰어다닌다
메타데이터: {'id': 3, 'source': 'loc_example'}
내용: 강아지가 주인을 보고 꼬리를 흔든다
메타데이터: {'id': 4, 'source': 'loc_example'}
내용: 호랑이가 숲속을 걸어 다닌다
메타데이터: {'id': 5, 'source': 'loc_example'}
내용: 사자가 초원에서 사냥을 준비한다
메타데이터: {'id': 6, 'source': 'loc_example'}
내용: 코끼리가 긴 코로 나뭇가지를 잡는다
메타데이터: {'id': 7, 'source': 'loc_example'}
내용: 기린이 나무 위 잎을 먹는다
메타데이터: {'id': 8, 'source': 'loc_example'}
내용: 원숭이가 바나나를 먹는다
메타데이터: {'id': 9, 'source': 'loc_example'}
내용: 펭귄이 얼음 위에서 걸어간다
메타데이터: {'id': 10, 'source': 'loc_example'}
내용: 사과가 빨갛게 익었다
메타데이터: {'id': 11, 'source': 'loc_example'}
내용: 바나나가 노랗게 잘 익어 달콤하다
메타데이터: {'id': 12, 'source': 'loc_example'}
내용: 포도가 달콤하고 신선하다
메타데이터: {'id': 13, 'source': 'loc_example'}
내용: 빵이 오븐에서 갓 구워졌다
메타데이터: {'id': 14, 'source': 'loc_example'}
내용: 라면이 뜨겁게 끓고 있다
메타데이터: {'id': 15, 'source': 'loc_example'}
내용: 김치찌개에서 얼큰한 냄새가 난다
메타데이터: {'id': 16, 'source': 'loc

In [5]:
# 임베딩 모델 생성
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

# db 확인 및 생성
# 기존 db 삭제
db_path = Path('./chroma_db')
if db_path.exists():
    shutil.rmtree(db_path)

vector_db = Chroma(
    collection_name="loc_docs",
    embedding_function=embedding_model,
    persist_directory=db_path,
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

In [12]:
#고유 ID 생성
document_ids = [f'doc_{document.metadata["id"]}' for document in documents]

# add_documents() => langchin이 내부에서 작업을 수행
# - page_content를 openAIEmbeddings 모델에 전달하여 벡터화
# - 벡터화된 데이터를 Chroma Db에 저장
# - 원본 문서와 메타데이터 저장

saved_ids = vector_db.add_documents(
    documents=documents,
    ids=document_ids
)

In [15]:
len(saved_ids)

30

In [20]:
# 사용자 검색 문장 입력 및 유사 문서 검색

query = input("검색 문장을 입려갛세요: ").strip()

res = vector_db._similarity_search_with_relevance_scores(
    query=query,
    k=3
)

In [22]:
print(query)
for doc, score in res:
    print(f'내용: {doc.page_content}')
    print(f'메타데이터: {doc.metadata}')
    print(f'유사도 점수: {score}')
    print('='*20)


내용: 빵이 오븐에서 갓 구워졌다
메타데이터: {'source': 'loc_example', 'id': 14}
유사도 점수: 0.12963616847991943
내용: 뉴욕의 자유의 여신상이 우뚝 서 있다
메타데이터: {'id': 26, 'source': 'loc_example'}
유사도 점수: 0.11969339847564697
내용: 고양이가 창가에서 햇볕을 쬐고 있다
메타데이터: {'id': 1, 'source': 'loc_example'}
유사도 점수: 0.11747288703918457
